# SDH exp_014 — focal LGBM and pair specialists
exp13 피처를 고정하고 메인 loss, pair specialist, LR blend를 순서대로 비교한다. 각 셀은 독립적인 실험 case다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'experiments').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'experiments').exists():
    raise RuntimeError('저장소 내부에서 실행해 주세요.')
EXP_DIR = PROJECT_ROOT / 'experiments' / 'SDH' / 'exp_014_focal_lgbm_specialists'
RESULTS_DIR = EXP_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))
import lgbm_experiment as exp
print('project root:', PROJECT_ROOT)

In [ ]:
data_dir = PROJECT_ROOT / 'data' / 'raw'
train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
genes = [column for column in train if column not in ('ID', 'SUBCLASS')]
labels = train['SUBCLASS'].to_numpy()
MAIN_CASES = exp.main_cases()
SPECIALIST_CASES = exp.specialist_cases()
display(pd.DataFrame([vars(case) for case in MAIN_CASES.values()]))
display(pd.DataFrame([vars(case) for case in SPECIALIST_CASES.values()]))

## 1. Seed 42 fold 피처 준비
가장 오래 걸리는 전처리를 fold별로 한 번만 수행한다. 이후 모든 모델 case가 같은 행렬을 재사용한다.

In [ ]:
prepared_42, labels_42, classes_42 = exp.prepare_seed(train, genes, seed=42)
assert np.array_equal(labels, labels_42)
print('prepared folds:', len(prepared_42))
print('feature counts:', [fold.feature_count for fold in prepared_42])

## 2. 공통 LR 기준

In [ ]:
results_42 = {}
lr_42 = exp.evaluate_lr_reference(prepared_42, labels_42, classes_42, seed=42)
results_42[lr_42.name] = lr_42
display(pd.DataFrame([lr_42.summary]))

## 3. 메인 LGBM 5개 case

In [ ]:
case_name = 'main_01_multiclass_balanced'
results_42[case_name] = exp.evaluate_main_case(prepared_42, labels_42, classes_42, MAIN_CASES[case_name], seed=42)
display(pd.DataFrame([results_42[case_name].summary]))

In [ ]:
case_name = 'main_02_multiclass_unweighted'
results_42[case_name] = exp.evaluate_main_case(prepared_42, labels_42, classes_42, MAIN_CASES[case_name], seed=42)
display(pd.DataFrame([results_42[case_name].summary]))

In [ ]:
case_name = 'main_03_focal_g1'
results_42[case_name] = exp.evaluate_main_case(prepared_42, labels_42, classes_42, MAIN_CASES[case_name], seed=42)
display(pd.DataFrame([results_42[case_name].summary]))

In [ ]:
case_name = 'main_04_focal_g2'
results_42[case_name] = exp.evaluate_main_case(prepared_42, labels_42, classes_42, MAIN_CASES[case_name], seed=42)
display(pd.DataFrame([results_42[case_name].summary]))

In [ ]:
case_name = 'main_05_focal_g1_balanced'
results_42[case_name] = exp.evaluate_main_case(prepared_42, labels_42, classes_42, MAIN_CASES[case_name], seed=42)
display(pd.DataFrame([results_42[case_name].summary]))

In [ ]:
main_leaderboard = exp.summaries(results_42)
main_leaderboard.to_csv(RESULTS_DIR / 'main_seed42_leaderboard.csv', index=False)
display(main_leaderboard)

## 4. 메인 승자 선택 및 specialist 확률 학습
LR을 제외한 메인 LGBM 중 seed42 Macro F1 최고를 자동 선택한다. 필요하면 SELECTED_MAIN을 직접 고정한 뒤 다음 셀을 실행한다.

In [ ]:
SELECTED_MAIN = max(
    (name for name in results_42 if name.startswith('main_')),
    key=lambda name: results_42[name].summary['oof_f1_macro'],
)
selected_main_42 = results_42[SELECTED_MAIN]
specialist_probability_42 = exp.fit_specialist_probabilities(prepared_42, labels_42, seed=42)
[(item.fold, item.pairs) for item in specialist_probability_42]
print('selected main:', SELECTED_MAIN, selected_main_42.summary['oof_f1_macro'])

## 5. Specialist 7개 case

In [ ]:
specialist_results_42 = {}
case_name = 'spec_01_rank1_soft_mass_030'
specialist_results_42[case_name] = exp.apply_specialist_case(selected_main_42, specialist_probability_42, labels_42, SPECIALIST_CASES[case_name])
display(pd.DataFrame([specialist_results_42[case_name].summary]))

In [ ]:
case_name = 'spec_02_rank2_soft_mass_030'
specialist_results_42[case_name] = exp.apply_specialist_case(selected_main_42, specialist_probability_42, labels_42, SPECIALIST_CASES[case_name])
display(pd.DataFrame([specialist_results_42[case_name].summary]))

In [ ]:
case_name = 'spec_03_both_soft_mass_015'
specialist_results_42[case_name] = exp.apply_specialist_case(selected_main_42, specialist_probability_42, labels_42, SPECIALIST_CASES[case_name])
display(pd.DataFrame([specialist_results_42[case_name].summary]))

In [ ]:
case_name = 'spec_04_both_soft_mass_030'
specialist_results_42[case_name] = exp.apply_specialist_case(selected_main_42, specialist_probability_42, labels_42, SPECIALIST_CASES[case_name])
display(pd.DataFrame([specialist_results_42[case_name].summary]))

In [ ]:
case_name = 'spec_05_both_soft_mass_050'
specialist_results_42[case_name] = exp.apply_specialist_case(selected_main_42, specialist_probability_42, labels_42, SPECIALIST_CASES[case_name])
display(pd.DataFrame([specialist_results_42[case_name].summary]))

In [ ]:
case_name = 'spec_06_both_soft_predicted_030'
specialist_results_42[case_name] = exp.apply_specialist_case(selected_main_42, specialist_probability_42, labels_42, SPECIALIST_CASES[case_name])
display(pd.DataFrame([specialist_results_42[case_name].summary]))

In [ ]:
case_name = 'spec_07_both_hard_predicted'
specialist_results_42[case_name] = exp.apply_specialist_case(selected_main_42, specialist_probability_42, labels_42, SPECIALIST_CASES[case_name])
display(pd.DataFrame([specialist_results_42[case_name].summary]))

In [ ]:
specialist_leaderboard = exp.summaries(specialist_results_42)
specialist_leaderboard.to_csv(RESULTS_DIR / 'specialist_seed42_leaderboard.csv', index=False)
display(specialist_leaderboard)

## 6. LR 대비 예측 다양성

In [ ]:
all_lgbm_results_42 = {**{k: v for k, v in results_42.items() if k.startswith('main_')}, **specialist_results_42}
diversity = pd.DataFrame([exp.diversity_metrics(lr_42, result, labels_42) for result in all_lgbm_results_42.values()]).sort_values('oracle_f1_macro', ascending=False)
diversity.to_csv(RESULTS_DIR / 'diversity_vs_exp13_lr_seed42.csv', index=False)
display(diversity)

## 7. 최고 후보와 LR 고정 weight blend

In [ ]:
SELECTED_CANDIDATE = max(all_lgbm_results_42, key=lambda name: all_lgbm_results_42[name].summary['oof_f1_macro'])
candidate_42 = all_lgbm_results_42[SELECTED_CANDIDATE]
blend_42 = exp.fixed_blends(lr_42, candidate_42, labels_42)
blend_42.insert(0, 'candidate', SELECTED_CANDIDATE)
blend_42.to_csv(RESULTS_DIR / 'lr_lgbm_blend_seed42.csv', index=False)
print('selected candidate:', SELECTED_CANDIDATE)
display(blend_42)

## 8. 클래스별 비교
LR, LGBM 후보, seed42 최고 blend의 클래스별 F1을 확인한다.

In [ ]:
best_blend_row = blend_42.sort_values('f1_macro', ascending=False).iloc[0]
best_model_weight = float(best_blend_row['model_weight'])
best_blend_probability = (1-best_model_weight)*lr_42.probability + best_model_weight*candidate_42.probability
best_blend_prediction = classes_42[best_blend_probability.argmax(axis=1)]
class_rows = []
for name, prediction in [('exp13_lr', lr_42.prediction), (SELECTED_CANDIDATE, candidate_42.prediction), ('best_blend', best_blend_prediction)]:
    report = classification_report(labels_42, prediction, labels=classes_42, output_dict=True, zero_division=0)
    for class_name in classes_42:
        class_rows.append({'model': name, 'class': class_name, **report[class_name]})
class_metrics = pd.DataFrame(class_rows)
class_metrics.to_csv(RESULTS_DIR / 'class_metrics_seed42.csv', index=False)
display(class_metrics.pivot(index='class', columns='model', values='f1-score'))

In [ ]:
lr_oof = exp.oof_probability_frame(lr_42, train['ID'], train['SUBCLASS'], prepared_42)
candidate_oof = exp.oof_probability_frame(candidate_42, train['ID'], train['SUBCLASS'], prepared_42)
lr_oof.to_csv(RESULTS_DIR / 'oof_exp13_lr_seed42.csv', index=False)
candidate_oof.to_csv(RESULTS_DIR / f'oof_{SELECTED_CANDIDATE}_seed42.csv', index=False)
print('OOF probabilities saved; class order:', list(classes_42))

## 9. 3-seed confirmation
아래 세 값을 seed42 결과로 잠근 뒤 실행한다. specialist를 사용하지 않는 경우 CONFIRM_SPECIALIST_CASE=None으로 둔다. weight도 seed마다 다시 선택하지 않는다.

In [ ]:
CONFIRM_MAIN_CASE = SELECTED_MAIN
CONFIRM_SPECIALIST_CASE = SELECTED_CANDIDATE if SELECTED_CANDIDATE.startswith('spec_') else None
LOCKED_MODEL_WEIGHT = best_model_weight
print(CONFIRM_MAIN_CASE, CONFIRM_SPECIALIST_CASE, LOCKED_MODEL_WEIGHT)

In [ ]:
confirmation_rows = []
confirmation_objects = {}
for seed in (42, 52, 62):
    if seed == 42:
        prepared, seed_labels, seed_classes = prepared_42, labels_42, classes_42
        lr_result = lr_42
        main_result = results_42[CONFIRM_MAIN_CASE]
        if CONFIRM_SPECIALIST_CASE is None:
            model_result = main_result
        else:
            model_result = specialist_results_42[CONFIRM_SPECIALIST_CASE]
    else:
        prepared, seed_labels, seed_classes = exp.prepare_seed(train, genes, seed=seed)
        lr_result = exp.evaluate_lr_reference(prepared, seed_labels, seed_classes, seed=seed)
        main_result = exp.evaluate_main_case(prepared, seed_labels, seed_classes, MAIN_CASES[CONFIRM_MAIN_CASE], seed=seed)
        if CONFIRM_SPECIALIST_CASE is None:
            model_result = main_result
        else:
            specialist_probability = exp.fit_specialist_probabilities(prepared, seed_labels, seed=seed)
            model_result = exp.apply_specialist_case(main_result, specialist_probability, seed_labels, SPECIALIST_CASES[CONFIRM_SPECIALIST_CASE])
    blend_probability = (1-LOCKED_MODEL_WEIGHT)*lr_result.probability + LOCKED_MODEL_WEIGHT*model_result.probability
    blend_prediction = seed_classes[blend_probability.argmax(axis=1)]
    blend_f1 = f1_score(seed_labels, blend_prediction, average='macro')
    confirmation_rows.append({
        'seed': seed,
        'lr_f1': lr_result.summary['oof_f1_macro'],
        'model_f1': model_result.summary['oof_f1_macro'],
        'blend_f1': blend_f1,
        'blend_delta_vs_lr': blend_f1-lr_result.summary['oof_f1_macro'],
        'model_weight': LOCKED_MODEL_WEIGHT,
    })
    confirmation_objects[seed] = {'lr': lr_result, 'model': model_result}
confirmation = pd.DataFrame(confirmation_rows)
confirmation.to_csv(RESULTS_DIR / 'confirmation_3seed.csv', index=False)
display(confirmation)
print('mean blend delta:', confirmation['blend_delta_vs_lr'].mean())
print('improved seeds:', int((confirmation['blend_delta_vs_lr'] > 0).sum()), '/ 3')

## 10. 제출 파일 생성 — safe LR 80% + dynamic specialist LGBM 20%, 3-seed 평균
위 실험에서 확정한 exp14 챔피언을 전체 train으로 다시 학습한다. 피처 통계와 유사 class pair는 train에서만 정하고, test는 transform과 predict에만 사용한다. 아래 셀을 순서대로 실행하면 `results/submission_exp014_safe_lr80_lgbm20_3seed.csv`가 생성된다.

In [ ]:
from lightgbm import LGBMClassifier

EXP13_DIR = PROJECT_ROOT / 'experiments' / 'SDH' / 'exp_013_standalone_pipeline_audit'
if str(EXP13_DIR) not in sys.path:
    sys.path.insert(0, str(EXP13_DIR))
import standalone_pipeline as p13

SUBMISSION_SEEDS = (42, 52, 62)
LR_WEIGHT, LGBM_WEIGHT = 0.80, 0.20
SUBMISSION_MAIN_CASE = 'main_01_multiclass_balanced'

def aligned_probability(model, matrix, classes):
    raw = np.asarray(model.predict_proba(matrix), dtype=np.float64)
    lookup = {name: index for index, name in enumerate(model.classes_)}
    probability = raw[:, [lookup[name] for name in classes]]
    np.testing.assert_allclose(probability.sum(axis=1), 1.0, atol=1e-6)
    return probability

def discover_similar_pairs(matrix, names, y, top_n=2):
    gene_columns = np.array([name.startswith('G__') for name in names])
    if not gene_columns.any():
        raise ValueError('G__ mutation-gene columns가 없습니다.')
    gene_matrix = matrix[:, gene_columns]
    class_names = sorted(np.unique(y))
    centroids = []
    for class_name in class_names:
        centroid = np.asarray(gene_matrix[y == class_name].mean(axis=0)).ravel()
        norm = np.linalg.norm(centroid)
        centroids.append(centroid / norm if norm > 0 else centroid)
    candidates = []
    for left_index, left in enumerate(class_names):
        for right_index in range(left_index + 1, len(class_names)):
            similarity = float(centroids[left_index] @ centroids[right_index])
            candidates.append((-similarity, left, class_names[right_index]))
    candidates.sort()
    return tuple((left, right) for _, left, right in candidates[:top_n])

def fit_binary_specialist(x_train, y, x_test, pair, seed):
    pair_mask = np.isin(y, pair)
    model = LGBMClassifier(
        objective='binary', boosting_type='gbdt', reg_alpha=0.0, reg_lambda=0.0,
        importance_type='gain', class_weight='balanced', random_state=seed,
        n_jobs=-1, deterministic=True, force_col_wise=True, verbosity=-1,
        **exp._specialist_parameters(seed),
    )
    model.fit(x_train[pair_mask], y[pair_mask])
    raw = np.asarray(model.predict_proba(x_test), dtype=np.float64)
    lookup = {name: index for index, name in enumerate(model.classes_)}
    return raw[:, [lookup[name] for name in pair]]

def apply_hard_routing(main_probability, classes, pairs, specialist_probabilities):
    probability = np.asarray(main_probability, dtype=np.float64).copy()
    original_prediction = classes[np.argmax(main_probability, axis=1)]
    class_lookup = {name: index for index, name in enumerate(classes)}
    routed = np.zeros(len(probability), dtype=bool)
    for pair, specialist in zip(pairs, specialist_probabilities):
        columns = [class_lookup[pair[0]], class_lookup[pair[1]]]
        pair_mass = probability[:, columns].sum(axis=1)
        mask = np.isin(original_prediction, pair)
        probability[mask, columns[0]] = pair_mass[mask] * specialist[mask, 0]
        probability[mask, columns[1]] = pair_mass[mask] * specialist[mask, 1]
        routed |= mask
    np.testing.assert_allclose(probability.sum(axis=1), 1.0, atol=1e-6)
    return probability, int(routed.sum())

In [ ]:
sample_submission = pd.read_csv(data_dir / 'sample_submission.csv')
classes = np.asarray(sorted(np.unique(labels)))
assert list(test.columns) == ['ID', *genes]
assert len(sample_submission) == len(test)
assert sample_submission['ID'].equals(test['ID'])
assert not train['ID'].duplicated().any()
assert not test['ID'].duplicated().any()

seed_test_probabilities = []
submission_audit = []
main_case = MAIN_CASES[SUBMISSION_MAIN_CASE]
for seed in SUBMISSION_SEEDS:
    print(f'\n===== submission seed {seed} =====')
    x_train, x_test, feature_names, feature_audit = p13.build_design_matrices(
        train[genes], test[genes], labels, genes, seed=seed, use_fixed_contrast=False,
    )
    assert feature_audit['raw_train_test_concat'] is False
    assert not any(name.startswith(('C__', 'D__exact_')) for name in feature_names)

    lr_model = p13.make_model(seed)
    lr_model.fit(x_train, labels)
    lr_probability = aligned_probability(lr_model, x_test, classes)

    main_model = LGBMClassifier(**exp._main_parameters(seed, main_case, len(classes)))
    main_model.fit(x_train, labels)
    main_probability = exp._aligned_probability(main_model, x_test, classes, focal=False)

    pairs = discover_similar_pairs(x_train, feature_names, labels, top_n=2)
    specialist_probabilities = tuple(
        fit_binary_specialist(x_train, labels, x_test, pair, seed) for pair in pairs
    )
    routed_probability, routed_rows = apply_hard_routing(
        main_probability, classes, pairs, specialist_probabilities,
    )
    blended_probability = LR_WEIGHT * lr_probability + LGBM_WEIGHT * routed_probability
    np.testing.assert_allclose(blended_probability.sum(axis=1), 1.0, atol=1e-6)
    seed_test_probabilities.append(blended_probability)
    submission_audit.append({
        'seed': seed, 'feature_count': len(feature_names),
        'train_selected_pairs': pairs, 'hard_routed_test_rows': routed_rows,
    })
    print(f'features={len(feature_names):,} pairs={pairs} routed_rows={routed_rows:,}')

final_probability = np.mean(seed_test_probabilities, axis=0)
np.testing.assert_allclose(final_probability.sum(axis=1), 1.0, atol=1e-6)
submission = sample_submission.copy()
submission['SUBCLASS'] = classes[np.argmax(final_probability, axis=1)]
assert not submission['SUBCLASS'].isna().any()
submission_path = RESULTS_DIR / 'submission_exp014_safe_lr80_lgbm20_3seed.csv'
submission.to_csv(submission_path, index=False)
display(pd.DataFrame(submission_audit))
display(submission.head())
print('saved:', submission_path)